# Table-CNN MRC Colab runner — continuous prefix + LoRA
Use an H100 or another high-memory NVIDIA runtime. This runs the recommended one-decoder continuous-prefix model with LoRA and no cross-attention. Store a Hugging Face token under the `HF_TOKEN` key in Colab Secrets if your environment requires one.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd {REPO_DIR}
!python -m pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    os.environ["HF_TOKEN"] = token

## Verify the full model path

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/smoke_test.py --config configs/continuous_prefix_lora.yaml

## Train (run after the smoke test passes)

In [ ]:
DRIVE_OUTPUT = "/content/drive/MyDrive/cnn_qwen_table_mcr/outputs/continuous_prefix_lora"
!PYTHONUNBUFFERED=1 python -u scripts/run_experiment.py --config configs/continuous_prefix_lora.yaml --mirror-output-dir "{DRIVE_OUTPUT}"

## Diagnose the trained LoRA checkpoint
This compares correct versus shuffled tables on 200 validation examples using the best continuous-prefix LoRA checkpoint.

In [ ]:
DRIVE_SWEEP_ROOT = "/content/drive/MyDrive/cnn_qwen_table_mcr/outputs"
DIAGNOSTIC_OUTPUT = f"{DRIVE_SWEEP_ROOT}/diagnostics/continuous_prefix_lora"
!PYTHONUNBUFFERED=1 python -u scripts/diagnose_saved_runs.py --configs configs/continuous_prefix_lora.yaml --mirror-root "{DRIVE_SWEEP_ROOT}" --max-examples 200 --modes trained --checkpoint best --output-dir "{DIAGNOSTIC_OUTPUT}"